## objectives
- Create a pytorch dataset from the squared image folder
- Load imagent1k weights for resnet50 as a baseline

In [2]:
from collections import Counter

import os
import torch
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
from torchvision.transforms import InterpolationMode


In [3]:
data_dir = '/home/takayuki/Desktop/summer2025/plants/data/AquaticPlantLabData/squared'

train_percent = 0.8
batch_size = 1

random_seed = 8

In [4]:
torch.manual_seed(random_seed)

aug_tf = transforms.Compose([
    transforms.Resize((224, 224)),
    
    transforms.RandomResizedCrop(size=(224, 224), scale=(0.5, 1.0), ratio=(1.0, 1.0)),
    
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomVerticalFlip(p=0.5),
    transforms.RandomRotation(degrees=10),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.15, hue=0.01),
    transforms.RandomPerspective(distortion_scale=0.15, p=0.5, interpolation=InterpolationMode.BICUBIC),    
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

dataset = datasets.ImageFolder(data_dir, transform = aug_tf)

train_size = int(train_percent * len(dataset))
test_size = len(dataset) - train_size

train_dataset, test_dataset = torch.utils.data.random_split(dataset, [train_size, test_size], 
                                                              generator=torch.Generator().manual_seed(random_seed))
train_dataloader = DataLoader(train_dataset, batch_size, shuffle=True)
test_dataloader = DataLoader(test_dataset, batch_size, shuffle=False)

print(f"Train dataset size: {len(train_dataset)}")
print(f"Test dataset size: {len(test_dataset)}\n")

# print how many images and of which classes are in the train and test datasets

def count_classes(subset):
    labels = [subset.dataset.targets[i] for i in subset.indices]
    label_counts = Counter(labels)
    return label_counts

train_class_counts = count_classes(train_dataset)
test_class_counts = count_classes(test_dataset)
class_names = dataset.classes
max_len = max(len(name) for name in class_names)
for i, class_name in enumerate(class_names):
    train_count = train_class_counts.get(i, 0)
    test_count = test_class_counts.get(i, 0)
    class_name = class_name.ljust(max_len+1)
    print(f"{class_name}: Train: {train_count}\t, Test: {test_count}")


Train dataset size: 172
Test dataset size: 43

Brasenia schreberi       : Train: 6	, Test: 3
Cabomba                  : Train: 2	, Test: 3
Ceratophyllum demersum   : Train: 21	, Test: 0
Elodea canadensis        : Train: 11	, Test: 4
Heteranthera dubia       : Train: 6	, Test: 1
Hydrocharis morsus-ranae : Train: 8	, Test: 4
Myriophyllum sibiricum   : Train: 4	, Test: 0
Myriophyllum spicatum    : Train: 12	, Test: 1
Najas flexilis           : Train: 4	, Test: 1
Nitellopsis obtusa       : Train: 3	, Test: 1
Nuphar variegata         : Train: 3	, Test: 0
Nymphaea odorata         : Train: 2	, Test: 0
Potamogeton crispus      : Train: 15	, Test: 6
Potamogeton gramineus    : Train: 13	, Test: 4
Potamogeton illinoensis  : Train: 29	, Test: 6
Potamogeton natans       : Train: 8	, Test: 1
Potamogeton praelongus   : Train: 3	, Test: 0
Potamogeton richardsonii : Train: 4	, Test: 2
Potamogeton robbinsii    : Train: 7	, Test: 2
Ranunculus aquatilis     : Train: 3	, Test: 2
Vallisneria americana    : 

## Load pretrained model
- ResNet50 with ImageNet1K weights

In [5]:
from tqdm import tqdm
from datetime import datetime

import timm
import torch.nn as nn
from torch.utils.tensorboard import SummaryWriter
from torchinfo import summary


In [6]:
model = timm.create_model('resnet50', pretrained=True)
summary(model, input_size=(1, 3, 224, 224))

Layer (type:depth-idx)                   Output Shape              Param #
ResNet                                   [1, 1000]                 --
├─Conv2d: 1-1                            [1, 64, 112, 112]         9,408
├─BatchNorm2d: 1-2                       [1, 64, 112, 112]         128
├─ReLU: 1-3                              [1, 64, 112, 112]         --
├─MaxPool2d: 1-4                         [1, 64, 56, 56]           --
├─Sequential: 1-5                        [1, 256, 56, 56]          --
│    └─Bottleneck: 2-1                   [1, 256, 56, 56]          --
│    │    └─Conv2d: 3-1                  [1, 64, 56, 56]           4,096
│    │    └─BatchNorm2d: 3-2             [1, 64, 56, 56]           128
│    │    └─ReLU: 3-3                    [1, 64, 56, 56]           --
│    │    └─Conv2d: 3-4                  [1, 64, 56, 56]           36,864
│    │    └─BatchNorm2d: 3-5             [1, 64, 56, 56]           128
│    │    └─Identity: 3-6                [1, 64, 56, 56]           --
│ 

In [7]:
print(model)

ResNet(
  (conv1): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
  (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (act1): ReLU(inplace=True)
  (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
  (layer1): Sequential(
    (0): Bottleneck(
      (conv1): Conv2d(64, 64, kernel_size=(1, 1), stride=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (act1): ReLU(inplace=True)
      (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (drop_block): Identity()
      (act2): ReLU(inplace=True)
      (aa): Identity()
      (conv3): Conv2d(64, 256, kernel_size=(1, 1), stride=(1, 1), bias=False)
      (bn3): BatchNorm2d(256, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
     

In [25]:
model.fc = nn.Linear(model.fc.in_features, len(dataset.classes))

for param in model.parameters():
    param.requires_grad = False
    
for param in model.fc.parameters():
    param.requires_grad = True
    
print({"Number of trainable parameters: ", sum(p.numel() for p in model.parameters() if p.requires_grad)})

{43029, 'Number of trainable parameters: '}


## Training Loop
- Tensorboard logging
- Save model checkpoints
- Save best models

In [26]:
# Parameters
run_name = "higher_lr_resnet50_augmented"
learning_rate = 0.005
num_epochs = 50

checkpoint_interval = 5
validation_interval = 1

# Paths
checkpoint_path = "/home/takayuki/Desktop/summer2025/plants/training/checkpoints"
models_path = "/home/takayuki/Desktop/summer2025/plants/training/models"
base_logs_path = "/home/takayuki/Desktop/summer2025/plants/training/runs/resnet50"

os.makedirs(checkpoint_path, exist_ok=True)
os.makedirs(models_path, exist_ok=True)
os.makedirs(base_logs_path, exist_ok=True)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model.to(device)
print(f"Using device: {device}, {torch.cuda.get_device_name(device) if device.type == 'cuda' else 'CPU'}")

Using device: cuda, NVIDIA RTX A500 Laptop GPU


In [27]:
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.fc.parameters(), lr=learning_rate)

now = datetime.now()
timestamp = now.strftime("%m-%d_%H-%M--%S")

model_name = model.name if hasattr(model, 'name') else model.__class__.__name__

hparams_dict = {
    "a_run_name": run_name,
    "learning_rate": learning_rate,
    "num_epochs": num_epochs,
    "batch_size": batch_size,
    "train_dataset_size": len(train_dataset),
    "test_dataset_size": len(test_dataset),
    "model_architecture": model_name,
    "optimizer": optimizer.__class__.__name__,
    "criterion": criterion.__class__.__name__,
    "timestamp": timestamp,
}

writer = SummaryWriter(log_dir=os.path.join(base_logs_path, f"{timestamp}_{run_name}_{model_name}"))

for key, value in hparams_dict.items():
    writer.add_text(key, str(value))

# Training loop
best_val_accuracy = 0.0
best_val_epoch = 0

print("Starting training...")
for epoch in range(num_epochs):
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0
    
    for images, labels in tqdm(train_dataloader, desc=f"Epoch {epoch + 1}/{num_epochs}", unit="batch"):
        images, labels = images.to(device), labels.to(device)
        
        optimizer.zero_grad()
        
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        
        running_loss += loss.item() * images.size(0)
        _, predicted = torch.max(outputs.data, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()
    
    train_loss = running_loss / total
    train_accuracy = correct / total
    
    print(f"Training Loss: {train_loss:.4f}, Training Accuracy: {train_accuracy*100:.4f}%")
    
    writer.add_scalar('Loss/train', train_loss, epoch)
    writer.add_scalar('Accuracy/train', train_accuracy, epoch)
    
    # Validation phase
    if (epoch + 1) % validation_interval == 0:
        model.eval()  
        val_running_loss = 0.0
        val_correct = 0
        val_total = 0
        
        with torch.no_grad():  
            
            for val_images, val_labels in test_dataloader:
                val_images, val_labels = val_images.to(device), val_labels.to(device)
                
                val_outputs = model(val_images)
                val_loss = criterion(val_outputs, val_labels)
                
                val_running_loss += val_loss.item() * val_images.size(0)
                _, val_predicted = torch.max(val_outputs.data, 1)
                val_total += val_labels.size(0)
                val_correct += (val_predicted == val_labels).sum().item()
        
        val_loss = val_running_loss / val_total
        val_accuracy = val_correct / val_total

        print(f"Validation Loss: {val_loss:.4f}, Validation Accuracy: {val_accuracy*100:.4f}%")
        
        writer.add_scalar('Loss/validation', val_loss, epoch)
        writer.add_scalar('Accuracy/validation', val_accuracy, epoch)
        
        # best model
        if val_accuracy > best_val_accuracy:
            best_val_accuracy = val_accuracy
            best_val_epoch = epoch + 1
            
            # Delete previous best model if exists
            best_model_name = f"best_model_{model_name}_{timestamp}.pth"
            best_model_file_path = os.path.join(models_path, best_model_name)
            
            if os.path.exists(best_model_file_path):
                os.remove(best_model_file_path)
            
            # Save the new one
            torch.save(model.state_dict(), best_model_file_path)
            print(f"Saved new best model: {best_model_file_path}")
        
        
        
        
    # Checkpoint saving phase
    if (epoch + 1) % checkpoint_interval == 0:
        checkpoint_name = f"checkpoint_epoch_{epoch + 1}_{model_name}_{timestamp}.pth"
        checkpoint_file_path = os.path.join(checkpoint_path, checkpoint_name)
        
        torch.save({
            'epoch': epoch + 1,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'train_loss': train_loss, 
        }, checkpoint_file_path)
        
        print(f"Saved checkpoint: {checkpoint_file_path}")

print("\nTraining complete.")
  

Starting training...


Epoch 1/50: 100%|██████████| 172/172 [00:16<00:00, 10.38batch/s]


Training Loss: 3.5850, Training Accuracy: 10.4651%
Validation Loss: 3.6823, Validation Accuracy: 2.3256%
Saved new best model: /home/takayuki/Desktop/summer2025/plants/training/models/best_model_ResNet_06-17_20-35--53.pth


Epoch 2/50: 100%|██████████| 172/172 [00:18<00:00,  9.53batch/s]


Training Loss: 3.2641, Training Accuracy: 13.9535%
Validation Loss: 2.9058, Validation Accuracy: 13.9535%
Saved new best model: /home/takayuki/Desktop/summer2025/plants/training/models/best_model_ResNet_06-17_20-35--53.pth


...

Training Loss: 1.6663, Training Accuracy: 45.3488%
Validation Loss: 3.0627, Validation Accuracy: 13.9535%


Epoch 49/50: 100%|██████████| 172/172 [00:18<00:00,  9.40batch/s]


Training Loss: 1.7367, Training Accuracy: 48.2558%
Validation Loss: 3.5728, Validation Accuracy: 20.9302%


Epoch 50/50: 100%|██████████| 172/172 [00:18<00:00,  9.53batch/s]


Training Loss: 1.6339, Training Accuracy: 45.9302%
Validation Loss: 3.1717, Validation Accuracy: 11.6279%
Saved checkpoint: /home/takayuki/Desktop/summer2025/plants/training/checkpoints/checkpoint_epoch_50_ResNet_06-17_20-35--53.pth

Training complete.


In [28]:
observations = f"Val loss doesn't decrease much"
hparams_dict["observations"] = observations
writer.add_text('Observations', observations)

# Save the final model
final_model_name = f"final_model_{model_name}_{timestamp}.pth"
final_model_file_path = os.path.join(models_path, final_model_name)
torch.save(model.state_dict(), final_model_file_path)
print(f"Saved final model: {final_model_file_path}")

# Write Metrics and Hyperparameters to TensorBoard
metrics = {
    "epochs_trained": epoch + 1,
    "final_train_loss": train_loss,
    "final_train_accuracy": train_accuracy,
    "final_val_loss": val_loss,
    "final_val_accuracy": val_accuracy,
    "best_val_accuracy": best_val_accuracy,
    "best_val_epoch": best_val_epoch,
}
writer.add_hparams(hparams_dict, metrics)
writer.flush()
writer.close()  

Saved final model: /home/takayuki/Desktop/summer2025/plants/training/models/final_model_ResNet_06-17_20-35--53.pth
